**Ingest Results - Incremental**

Reads all JSON files from the `results/` folder in the batch landing path, adds metadata, and writes to `formula1_incr.bronze.results` partitioned by `batch_id`.

**Load config and helpers**

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze_helpers

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import functions as f

**Verify written data**

In [0]:
dbutils.widgets.text('p_batch_id','')
batch_id= dbutils.widgets.get('p_batch_id')

In [0]:
source_file = f'{loding_folder_path}/{batch_id}/results' # to replace the load data in read api
table_name = f'{catalog_name}.{bronze_schema}.results' # to replace the save table in write api 

**Verify**

**Define schema** (DDL string, 14 columns)

In [0]:
from pyspark.sql.types import *
results_schema = ' date DATE,raceName string, round int, season int, url string, constructorId string, driverId string, grid int, laps int, number int, points double, position int, positionText string, status string'

**Read JSON folder** (Spark reads all files automatically)

In [0]:
results_df = (
    spark.read
    .format('json')
    #.option('Headers',True)
    .schema(results_schema)
    .load(source_file)
)
display(results_df)


**Add metadata columns**

In [0]:
results_final_df = add_ingestion_metadata(results_df)


**Write to bronze Delta table** (overwrite per batch partition)

In [0]:
write_to_bronze(
    input_df = results_final_df,
    table_name = table_name,
    batch_id = batch_id
)

In [0]:
display(spark.table(table_name))
